# Whiteboard Problem 2

**题目**：推导Metropolis算法的马尔可夫链转移核 (Derive the transition kernel of the Markov chain for the Metropolis algorithm)。

**解答**：
设目标分布的概率密度函数为 $\pi(x)$，建议（提议）分布为 $q(y|x)$。在经典的 Metropolis 算法中，建议分布是对称的，即由于随机游走的过程，有 $q(y|x) = q(x|y)$。

Metropolis 算法的转移过程有两步：
1. 给定当前状态 $x$，从建议分布 $q(y|x)$ 中采样产生一个候选状态 $y$。
2. 以一定的接受概率 $\alpha(x, y)$ 接受这个候选状态：
   $$ \alpha(x, y) = \min\left(1, \frac{\pi(y)}{\pi(x)}\right) $$
   
整个马尔可夫链的转移核 $K(x, y)$ 可以分为两种情况讨论：

- **当候选状态被接受时 ($y \neq x$)**：转移到状态 $y$ 的概率密度由提出 $y$ 的概率和接受 $y$ 的概率共同决定：
  $$ K(x, y) = q(y|x) \alpha(x, y) $$

- **当候选状态被拒绝时 ($y = x$)**：如果提出的候选状态被拒绝，马尔可夫链会停留在原状态 $x$。所有可能引发拒绝的候选状态都会导致系统留在 $x$。因此，留在 $x$ 的概率是拒绝所有其他状态的概率之和（可以用示性/狄拉克 $\delta$ 函数表示为集中的概率质量）：
  $$ P(\text{拒绝}) = 1 - \int q(y'|x) \alpha(x, y') dy' $$

综上所述，Metropolis 算法的完整马尔可夫链转移核写为：
$$ K(x, y) = q(y|x)\alpha(x,y) + \left( 1 - \int q(y'|x)\alpha(x, y') dy' \right) \delta_x(y) $$

# Coding Lab 2

**题目**：在一个二维标准正态分布（根据讲义这里应该是均值为 $[10, 10]^T$，协方差矩阵为对角矩阵 $I = \begin{bmatrix}1 & 0 \\ 0 & 1\end{bmatrix}$）上运行简单的随机游走 Metropolis 算法。
返回其接受率 (acceptance ratio)，以及马尔可夫链的均值和方差。
假设链的执行步数为 1000 (`steps=1000`)，预热期步数为 50 (`burnIn=50`)。

In [1]:
import numpy as np

# -------------------- 参数设定 --------------------
mu = np.array([10.0, 10.0])
Sigma = np.array([[1.0, 0.0], 
                  [0.0, 1.0]])
n_steps = 1000
burn_in = 50
proposal_std = 1.0  # 随机游走提议分布的标准差

# 初始状态 (可以随机选择或固定)
current_x = np.array([0.0, 0.0]) 

# ---------------- 目标分布对数密度 ----------------
def log_target(x):
    """
    计算正态分布对数概率密度正比项:
    log_pi(x) ∝ -0.5 * (x - mu).T @ Sigma^-1 @ (x - mu)
    """
    diff = x - mu
    # 因为协方差是单位矩阵，直接算内积即可
    return -0.5 * np.sum(diff**2)

# -------------- Metropolis 算法主循环 --------------
samples = np.zeros((n_steps, 2))
accepted = 0

for i in range(n_steps):
    # 1. 给出随机游走的候选状态 q(y | x) ~ N(x, proposal_std^2 * I)
    proposed_y = current_x + np.random.normal(0, proposal_std, size=2)
    
    # 2. 计算接受概率 alpha
    # 因为是随机游走对称提议且采用对数运算： log_alpha = min(0, log_pi(y) - log_pi(x))
    log_alpha = log_target(proposed_y) - log_target(current_x)
    
    # 3. 决定是否接受候选状态
    # log(u) < log_alpha 等价于 u < alpha (u ~ U(0,1))
    if np.log(np.random.rand()) < log_alpha:
        current_x = proposed_y
        accepted += 1
        
    samples[i, :] = current_x

# -------------------- 结果分析 --------------------
# 去除预热期的样本 (Burn-In)
valid_samples = samples[burn_in:]

# 计算并返回结果
acceptance_ratio = accepted / n_steps
chain_mean = np.mean(valid_samples, axis=0)
chain_variance = np.var(valid_samples, axis=0)

print(f"总步数 (Total steps): {n_steps}")
print(f"预热期 (Burn-in steps): {burn_in}")
print(f"接受率 (Acceptance ratio): {acceptance_ratio:.4f}")
print(f"马尔可夫链均值 (Chain mean): {chain_mean}")
print(f"马尔可夫链方差 (Chain variance): {chain_variance}")

总步数 (Total steps): 1000
预热期 (Burn-in steps): 50
接受率 (Acceptance ratio): 0.5380
马尔可夫链均值 (Chain mean): [ 9.83426696 10.04780686]
马尔可夫链方差 (Chain variance): [1.01783548 0.92626191]


# 补充: Whiteboard Problem 2 (Gambler's Ruin)

**问题描述**：你有 ¥100，并且在明天早上前必须要凑齐 ¥1000。你唯一获得钱的途径就是赌博。如果你下注 $k$，你有 $p$ 的概率赢下 $k$，或者以 $1-p$ 的概率输掉 $k$。
- **最大化策略 (Maximal strategy)**: 每次下注尽可能多，最高不超过你凑齐 ¥1000 所需要的差额。
- **最小化策略 (Minimal strategy)**: 每次下注金额最小，例如 1。

(a) 如果 $p = 0.45$，哪种策略更好？  
(b) 如果 $p = 0.8$，哪种策略更好？

**解答**：
这是一个经典的赌徒破产问题（也称 Bold play vs Timid play）：
- **(a) 当 $p = 0.45 < 0.5$ 时（游戏对你不利 / Subfair game）**：由于每次赌博预期收益为负，你参与赌博的次数越多，根据大数定律你最终彻底输光（破产）的概率就越大。因此，为了最大化达到 ¥1000 的概率，你应该尽量**减少**赌博的次数。**最大化策略 (Maximal strategy / Bold play)** 是更好的选择。
- **(b) 当 $p = 0.8 > 0.5$ 时（游戏对你有利 / Superfair game）**：每次赌博预期收益为正。只要你有足够长的时间玩下去，资本积累的概率极高。此时你的主要风险是“由于短期连续输钱而导致提早破产离场”。为了平摊这种风险，应该把每次下注额控制在最小。因此在这个情况下，**最小化策略 (Minimal strategy / Timid play)** 是更好的选择。

In [3]:
import numpy as np

# 补充: Coding Lab 2
# 使用蒙特卡洛模拟(Monte Carlo simulation)在上述两种情况(p=0.45和p=0.8)下估计预期的收益和95%置信区间，印证理论推导。

def simulate_gamble(strategy, p, initial_wealth=100, target=1000):
    wealth = initial_wealth
    while wealth > 0 and wealth < target:
        if strategy == 'maximal':
            # 最大化策略：下注当前金额与目标差额之间的较小值
            bet = min(wealth, target - wealth)
        elif strategy == 'minimal':
            # 最小化策略：每次仅下注 1
            bet = 1
        else:
            raise ValueError("Unknown strategy")
        
        # 以概率 p 赢得 bet，否则输掉 bet
        if np.random.rand() < p:
            wealth += bet
        else:
            wealth -= bet
    return wealth

def monte_carlo_experiment(strategy, p, n_trials=1000):
    results = np.zeros(n_trials)
    for i in range(n_trials):
        results[i] = simulate_gamble(strategy, p)
    
    # 最终的实际收益(停留在1000或0)
    expected_return = np.mean(results)
    
    # 样本标准误差 (Standard Error)
    std_err = np.std(results, ddof=1) / np.sqrt(n_trials)
    
    # 使用正态近似求95%置信区间
    ci_lower = expected_return - 1.96 * std_err
    ci_upper = expected_return + 1.96 * std_err
    
    # 达到 1000 的概率 (即胜率)
    win_prob = np.mean(results == 1000)
    
    return expected_return, ci_lower, ci_upper, win_prob

# 运行两组模拟
np.random.seed(42) # 固定随机种子以保证结果可复现

scenarios = [
    ('maximal', 0.45),
    ('minimal', 0.45),
    ('maximal', 0.8),
    ('minimal', 0.8)
]

print("====== Monte Carlo Simulation Results (1000 trials each) ======\n")
for strategy, p in scenarios:
    exp_return, ci_low, ci_high, win_prob = monte_carlo_experiment(strategy, p, n_trials=1000)
    print(f"[{strategy.capitalize()} Strategy | p = {p}]")
    print(f"  胜率 (Win Probability) : {win_prob*100:.1f}%")
    print(f"  预期最终资金 (Expected Return) : ¥{exp_return:.2f}")
    print(f"  95% 置信区间 (95% CI)  : [¥{ci_low:.2f}, ¥{ci_high:.2f}]\n")

====== Monte Carlo Simulation Results (1000 trials each) ======

[Maximal Strategy | p = 0.45]
  胜率 (Win Probability) : 6.0%
  预期最终资金 (Expected Return) : ¥60.00
  95% 置信区间 (95% CI)  : [¥45.27, ¥74.73]

[Minimal Strategy | p = 0.45]
  胜率 (Win Probability) : 0.0%
  预期最终资金 (Expected Return) : ¥0.00
  95% 置信区间 (95% CI)  : [¥0.00, ¥0.00]

[Maximal Strategy | p = 0.8]
  胜率 (Win Probability) : 50.4%
  预期最终资金 (Expected Return) : ¥504.00
  95% 置信区间 (95% CI)  : [¥473.00, ¥535.00]

[Minimal Strategy | p = 0.8]
  胜率 (Win Probability) : 100.0%
  预期最终资金 (Expected Return) : ¥1000.00
  95% 置信区间 (95% CI)  : [¥1000.00, ¥1000.00]



# OpenCode visibility test
If you can see this markdown cell, OpenCode successfully edited the notebook file.

In [1]:
import sys
print('OpenCode can append cells to this notebook.')
print(sys.executable)
print(sys.version)


OpenCode can append cells to this notebook.
d:\miniconda3\envs\jupyter\python.exe
3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)]
